In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout

from sklearn.utils.class_weight import compute_class_weight

In [2]:
train_df = pd.read_csv("../dataset/train_split.csv")
val_df = pd.read_csv("../dataset/val_split.csv")

print("Jumlah training   :", len(train_df))
print("Jumlah validation :", len(val_df))

print("\nContoh data:")
print(train_df.head())

Jumlah training   : 21947
Jumlah validation : 5487

Contoh data:
                                          image_path    label  \
0  ../dataset/processed/train\neutral\Training_19...  neutral   
1  ../dataset/processed/train\happy\Training_5977...    happy   
2  ../dataset/processed/train\angry\Training_2226...    angry   
3  ../dataset/processed/train\angry\Training_5622...    angry   
4  ../dataset/processed/train\angry\Training_7324...    angry   

                               hash  label_index  
0  59e9cf176ae7903974f8e2445ef185c9            4  
1  2222a4629a48c362a95cbd5f5ad7fa74            3  
2  b3f7d04564912e590c91a62d5a10c179            0  
3  7331dc3297e9693ded4c6a56a1dd626d            0  
4  9421f59d2c9679c60e5289cc52e8f468            0  


In [3]:
class_names = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]

print(class_names)

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [4]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(range(len(class_names))),
    y=train_df["label_index"]
)

class_weight_dict = dict(
    enumerate(class_weights)
)

for index, weight in class_weight_dict.items():
    print(f"{class_names[index]:10} : {weight:.4f}")

angry      : 1.0226
disgust    : 10.3475
fear       : 1.0088
happy      : 0.5532
neutral    : 0.8052
sad        : 0.8310
surprise   : 1.4685


In [5]:
def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=1)

    image = tf.cast(image, tf.float32)
    image = image / 255.0

    return image, label

In [6]:
BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["image_path"].values,
        train_df["label_index"].values
    )
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].values,
        val_df["label_index"].values
    )
)

train_dataset = train_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.shuffle(
    len(train_df),
    seed=42
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [7]:
images, labels = next(iter(train_dataset))

print("Shape images :", images.shape)
print("Shape labels :", labels.shape)
print("Pixel min    :", tf.reduce_min(images).numpy())
print("Pixel max    :", tf.reduce_max(images).numpy())

Shape images : (32, 48, 48, 1)
Shape labels : (32,)
Pixel min    : 0.0
Pixel max    : 1.0


In [8]:
model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(48, 48, 1)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(7, activation="softmax")
])

model.summary()

d:\JamTerbang\DL\KlasifikasiEkspresi\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       819,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 839,047 (3.20 MB)

 Trainable params: 839,047 (3.20 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    class_weight=class_weight_dict
)

d:\JamTerbang\DL\KlasifikasiEkspresi\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 22s 25ms/step - accuracy: 0.1907 - loss: 1.9044 - val_accuracy: 0.2561 - val_loss: 1.8284
Epoch 2/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.3049 - loss: 1.7454 - val_accuracy: 0.3406 - val_loss: 1.6999
Epoch 3/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.3491 - loss: 1.6460 - val_accuracy: 0.3978 - val_loss: 1.5834
Epoch 4/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.3762 - loss: 1.5678 - val_accuracy: 0.4044 - val_loss: 1.5485
Epoch 5/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.4016 - loss: 1.4947 - val_accuracy: 0.3909 - val_loss: 1.5715
Epoch 6/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.4217 - loss: 1.4438 - val_accuracy: 0.4350 - val_loss: 1.4708
Epoch 7/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.4470 - loss: 1.3683 - val_accuracy: 0.4266 - val_loss: 1.4952
Epoch 8/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.4599 - loss: 1.3193 - 

In [11]:
y_true_cw = []
y_pred_cw = []

for images, labels in val_dataset:
    predictions = model.predict(images, verbose=0)

    predicted_labels = np.argmax(predictions, axis=1)

    y_true_cw.extend(labels.numpy())
    y_pred_cw.extend(predicted_labels)

y_true_cw = np.array(y_true_cw)
y_pred_cw = np.array(y_pred_cw)

print("Jumlah data validation:", len(y_true_cw))

Jumlah data validation: 5487


In [12]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true_cw,
        y_pred_cw,
        target_names=class_names
    )
)

              precision    recall  f1-score   support

       angry       0.38      0.42      0.40       767
     disgust       0.22      0.24      0.23        76
        fear       0.35      0.26      0.30       777
       happy       0.68      0.69      0.68      1417
     neutral       0.44      0.47      0.46       973
         sad       0.36      0.37      0.36       943
    surprise       0.62      0.60      0.61       534

    accuracy                           0.48      5487
   macro avg       0.43      0.44      0.43      5487
weighted avg       0.48      0.48      0.48      5487



## Kesimpulan Experiment Class Weight

Penggunaan class weight meningkatkan kemampuan model dalam mengenali kelas minoritas, terutama kelas Disgust. Recall Disgust meningkat dari 0.07 pada baseline menjadi 0.24 setelah menggunakan class weight, sedangkan F1-score meningkat dari 0.12 menjadi 0.23.

Namun, performa keseluruhan mengalami penurunan. Accuracy validation turun dari 0.51 menjadi 0.48 dan weighted F1 turun dari 0.50 menjadi 0.48. Macro F1 tetap berada di sekitar 0.43.

Dengan demikian, class weight memberikan keuntungan pada kelas minoritas, tetapi belum memberikan peningkatan performa keseluruhan pada konfigurasi CNN yang digunakan.